# 1. Define and explore SysML v2 models

`longeron` parses the OMG SysML v2 **textual notation** into a typed
Python object model. The parsers are generated with ANTLR from the
SysML v2 and KerML grammars, and they cover the full grammar.

**You will learn how to:**

- parse SysML v2 text into a model (`longeron.loads`);
- walk the model tree and read typed fields (`iter_tree`, `find`);
- author the same model programmatically, from dataclasses;
- load a multi-file workspace through the model cache (`longeron.load`).

**Prerequisites:** `pip install longeron`. This tutorial uses only core
features, so no extras are needed.

The cell below parses a small vehicle model.

In [ ]:
import longeron

model = longeron.loads("""
package Vehicles {
    doc /* A small demonstration model. */

    part def Wheel { attribute diameter : Real = 0.66; }

    part def Vehicle {
        attribute mass : Real = 1200.0;
        attribute maxMass : Real = 2000.0;
        part wheels : Wheel[4];
        assert constraint massLimit { mass <= maxMass }
    }
}
""")
model

## The model is a tree of dataclasses

Every element has a `kind`, a `qualified_name`, and typed fields.
Closed vocabularies (kinds, directions, visibilities, ...) are
`typing.Literal` aliases, so tooling can check them statically.

The first cell below walks the tree with `iter_tree`. The second looks
up elements by qualified name with `find` and reads their fields.

In [ ]:
for element in model.iter_tree():
    kind = getattr(element, "kind", type(element).__name__)
    print(
        f"{'  ' * len((element.qualified_name or '').split('::'))}"
        f"{kind:12s} {element.qualified_name or ''}"
    )

In [ ]:
vehicle = model.find("Vehicles::Vehicle")
mass = model.find("Vehicles::Vehicle::mass")
print("kind:      ", mass.kind)
print("types:     ", mass.types)
print("value expr:", mass.value.expr.to_text())
print("supers of Vehicle:", vehicle.supers)
print("doc:", model.find("Vehicles").doc)

## Models can be built programmatically

The same dataclasses are the authoring API, so no text is required.
Expressions come from `longeron.parse_expression`. The generated model
prints back as SysML text, which shows that both authoring routes meet
in the same object model.

In [ ]:
from longeron import model as M

pkg = M.Package(name="Generated")
sensor = M.Definition(kind="part", name="Sensor")
sensor.add(
    M.Usage(
        kind="attribute",
        name="rate",
        types=["Real"],
        value=M.FeatureValue(longeron.parse_expression("100.0 * 2")),
    ),
    M.Usage(
        kind="attribute",
        name="enabled",
        types=["Boolean"],
        value=M.FeatureValue(longeron.parse_expression("true")),
    ),
)
pkg.add(sensor)

generated = M.Model()
generated.add(pkg)
print(longeron.to_sysml(generated))

## Multi-file workspaces

`longeron.load()` accepts a single `.sysml` file, a `.json` export, or a
**directory**. A directory load merges every file under one root
namespace, so cross-file imports resolve. Built models are cached as
JSON, keyed on the source content plus a fingerprint of the generated
parser and builder code. A warm directory load is ~1000x faster than a
cold parse with the ANTLR Python runtime.

The cell below writes a two-file workspace, loads it as one model, and
calls a calc from one file that reads an attribute imported from the
other.

In [ ]:
import tempfile
from pathlib import Path

workspace = Path(tempfile.mkdtemp())
(workspace / "units.sysml").write_text("package Units { attribute gravity : Real = 9.81; }")
(workspace / "app.sysml").write_text("""
package App {
    private import Units::*;
    calc def Weight { in m : Real; return : Real = m * gravity; }
}
""")

merged = longeron.load(workspace)  # directory -> merged model
interp = longeron.Interpreter(merged)
interp.call("App::Weight", m=10.0)  # cross-file resolution works